# Notebook 5 – Data Validation




In [ ]:
import pandas as pd
import numpy as np

## 1. Example dataset
We will validate employee records. The sample intentionally includes invalid values so the rules can detect them.

In [ ]:
employees = pd.DataFrame({
    "employee_id": [1, 2, 3, 4, 5],
    "name": ["Asha", "Bala", "Charan", "Divya", "Ezhil"],
    "age": [25, -3, 41, 30, 22],
    "salary": [45000, 52000, -1000, 60000, 38000],
    "percentage": [88, 104, 72, 65, -2],
    "gender": ["Female", "Male", "Other", "Unknown", "Female"],
    "date_of_birth": ["2001-03-10", "1998-07-12", "1985-01-01", "2020-01-01", "2004-06-20"],
    "date_of_joining": ["2024-06-01", "2023-02-15", "2020-09-01", "2019-05-01", "2025-01-10"],
    "department_id": [10, 20, 30, 40, 10]
})
employees

## 2. Parse and validate data types
Convert dates with `errors='coerce'`; values that cannot be parsed become `NaT` and can then be flagged. Avoid assuming that a column has the correct type just because it looks numeric.

In [ ]:
employees["date_of_birth"] = pd.to_datetime(employees["date_of_birth"], errors="coerce")
employees["date_of_joining"] = pd.to_datetime(employees["date_of_joining"], errors="coerce")

print(employees.dtypes)

## 3. Build validation rules
- **Range:** age and salary cannot be negative; percentage must be between 0 and 100.
- **Type:** age and salary should be numeric.
- **Format:** email, phone number, or ID should follow the expected format.
- **Category:** values must belong to an accepted set.
- **Null:** required fields cannot be missing.
- **Business rule:** joining date cannot be before birth date.
- **Referential:** department ID must exist in the department reference table.
- **Date:** date fields must parse and obey chronological constraints.
- **Constraint:** employee ID should be unique.

In [ ]:
allowed_genders = {"Female", "Male", "Other"}
valid_department_ids = {10, 20, 30}

validation = pd.DataFrame(index=employees.index)
validation["age_invalid"] = employees["age"].isna() | (employees["age"] < 0)
validation["salary_invalid"] = employees["salary"].isna() | (employees["salary"] < 0)
validation["percentage_invalid"] = ~employees["percentage"].between(0, 100, inclusive="both")
validation["gender_invalid"] = ~employees["gender"].isin(allowed_genders)
validation["name_missing"] = employees["name"].isna() | employees["name"].astype(str).str.strip().eq("")
validation["date_parse_invalid"] = employees["date_of_birth"].isna() | employees["date_of_joining"].isna()
validation["joining_before_birth"] = employees["date_of_joining"] < employees["date_of_birth"]
validation["department_invalid"] = ~employees["department_id"].isin(valid_department_ids)
validation["employee_id_duplicate"] = employees["employee_id"].duplicated(keep=False)

validation

## 4. Format validation example
This illustrative email rule checks for a simple `name@domain.tld` shape. Production systems may need a dedicated validator; a pattern check does not prove that an email address exists.

In [ ]:
emails = pd.Series(["asha@example.com", "not-an-email", "bala@company.in", None])
email_pattern = r"^[^\s@]+@[^\s@]+\.[^\s@]+$"
email_valid = emails.fillna("").str.match(email_pattern)
pd.DataFrame({"email": emails, "valid_format": email_valid})

## 5. View invalid rows and summarize

In [ ]:
invalid_rows = employees.loc[validation.any(axis=1)].copy()
invalid_rows

In [ ]:
validation.sum().sort_values(ascending=False).rename("invalid_count").to_frame()

## 6. Reusable validation function
This function returns a table of validation flags. It does not delete or automatically repair records.

In [ ]:
def validate_employees(df: pd.DataFrame) -> pd.DataFrame:
    """Return boolean validation flags for employee records."""
    required = {"employee_id", "name", "age", "salary", "percentage",
                "gender", "date_of_birth", "date_of_joining", "department_id"}
    missing_columns = required - set(df.columns)
    if missing_columns:
        raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

    flags = pd.DataFrame(index=df.index)
    flags["age_invalid"] = df["age"].isna() | (df["age"] < 0)
    flags["salary_invalid"] = df["salary"].isna() | (df["salary"] < 0)
    flags["percentage_invalid"] = ~df["percentage"].between(0, 100, inclusive="both")
    flags["gender_invalid"] = ~df["gender"].isin({"Female", "Male", "Other"})
    flags["name_missing"] = df["name"].isna() | df["name"].astype(str).str.strip().eq("")
    flags["joining_before_birth"] = df["date_of_joining"] < df["date_of_birth"]
    flags["department_invalid"] = ~df["department_id"].isin({10, 20, 30})
    flags["employee_id_duplicate"] = df["employee_id"].duplicated(keep=False)
    return flags

validate_employees(employees)